In [7]:
import mne
import torch
import numpy as np
from pathlib import Path
from collections import Counter


class SleepPredictorV1:
    """睡眠分期预测器 V1"""
    
    def __init__(self, model_path, device='cuda', window_size=15):
        """
        参数:
            model_path: 模型权重文件路径
            device: 计算设备
            window_size: 时序窗口大小
        """
        self.device = device if torch.cuda.is_available() else 'cpu'
        self.window_size = window_size
        self.model = None
        self.model_path = model_path
        
        print(f"[初始化] 使用设备: {self.device}")
        print(f"[初始化] 窗口大小: {window_size}")
    
    def load_model(self, model_class):
        """加载模型权重"""
        print(f"[加载模型] 路径: {self.model_path}")
        
        if not Path(self.model_path).exists():
            raise FileNotFoundError(f"模型文件不存在: {self.model_path}")
        
        self.model = model_class(num_classes=5, window_size=self.window_size).to(self.device)
        self.model.load_state_dict(torch.load(self.model_path, map_location=self.device))
        self.model.eval()
        
        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"[加载模型] 成功加载，参数量: {total_params:,}")
    
    def preprocess(self, edf_path):
        """
        预处理 EDF 文件
        严格按照 process.py 的逻辑
        
        返回: (N, 1, 3000) numpy 数组
        """
        print(f"\n[预处理] 开始处理: {edf_path}")
        
        # 1. 读取原始数据
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        print(f"[预处理] 原始采样率: {raw.info['sfreq']} Hz")
        print(f"[预处理] 原始通道: {raw.ch_names}")
        print(f"[预处理] 原始时长: {raw.times[-1]/3600:.2f} 小时")
        
        # 2. 通道选择（优先 Fpz-Cz，否则取第一个 EEG）
        if 'EEG Fpz-Cz' in raw.ch_names:
            raw.pick_channels(['EEG Fpz-Cz'])
            print(f"[预处理] 选择通道: EEG Fpz-Cz")
        else:
            # 兼容其他数据集
            eeg_channels = [ch for ch in raw.ch_names if 'EEG' in ch.upper()]
            if eeg_channels:
                raw.pick_channels([eeg_channels[0]])
                print(f"[预处理] 选择通道: {eeg_channels[0]}")
            else:
                raw.pick_types(eeg=True)
                print(f"[预处理] 自动选择 EEG 通道")
        
        # 3. 重采样到 100Hz
        if raw.info['sfreq'] != 100:
            print(f"[预处理] 重采样: {raw.info['sfreq']} Hz -> 100 Hz")
            raw.resample(100, verbose=False)
        
        # 4. 带通滤波（与训练时保持一致）
        print(f"[预处理] 滤波: 0.3-35 Hz")
        raw.filter(l_freq=0.3, h_freq=35, method='iir', verbose=False)
        
        # 5. 切分为 30s Epochs
        data = raw.get_data()[0]  # (n_samples,)
        epoch_len = 3000  # 30s * 100Hz
        n_epochs = len(data) // epoch_len
        
        print(f"[预处理] 总采样点: {len(data)}")
        print(f"[预处理] Epoch 数量: {n_epochs}")
        
        if n_epochs == 0:
            raise ValueError("数据长度不足 30 秒，无法处理")
        
        epochs = []
        for i in range(n_epochs):
            epoch = data[i*epoch_len : (i+1)*epoch_len]
            
            # 6. Z-score 标准化（每个 epoch 独立）
            mean, std = epoch.mean(), epoch.std()
            if std < 1e-8:
                print(f"[警告] Epoch {i} 标准差过小，可能是平坦信号")
                std = 1e-8
            epoch = (epoch - mean) / std
            epochs.append(epoch)
        
        result = np.array(epochs).reshape(n_epochs, 1, 3000)  # (N, 1, 3000)
        print(f"[预处理] 输出形状: {result.shape}")
        return result
    
    def predict(self, epochs):
        """
        执行推理
        
        输入: (N, 1, 3000) numpy 数组
        输出: 预测结果字典
        """
        if self.model is None:
            raise RuntimeError("模型未加载，请先调用 load_model()")
        
        print(f"\n[推理] 开始预测，Epoch 数量: {len(epochs)}")
        
        half_window = self.window_size // 2
        results = []
        
        with torch.no_grad():
            for i in range(len(epochs)):
                # 构建时序窗口（边缘用 edge padding）
                start = max(0, i - half_window)
                end = min(len(epochs), i + half_window + 1)
                
                window = epochs[start:end]
                
                # 边缘填充到 window_size
                if len(window) < self.window_size:
                    pad_left = max(0, half_window - i)
                    pad_right = max(0, (i + half_window + 1) - len(epochs))
                    window = np.pad(window, ((pad_left, pad_right), (0, 0), (0, 0)), mode='edge')
                
                # 转为 Tensor (1, window_size, 1, 3000)
                x = torch.from_numpy(window).unsqueeze(0).float().to(self.device)
                
                # 预测（5 分类）
                output = self.model(x)
                pred = output.argmax(1).item()
                results.append(pred)
                
                # 每 100 个 epoch 打印一次进度
                if (i + 1) % 100 == 0:
                    print(f"[推理] 进度: {i+1}/{len(epochs)}")
        
        print(f"[推理] 完成，原始预测分布（5分类）:")
        print(f"  {Counter(results)}")
        
        # 映射 5 类 -> 4 类
        def map_5to4(label):
            """
            原始(5类): 0:W, 1:N1, 2:N2, 3:N3, 4:REM
            目标(4类): 0:W, 1:REM, 2:Light(N1+N2), 3:Deep(N3)
            """
            if label == 0: return 0  # W
            if label == 4: return 1  # REM
            if label in [1, 2]: return 2  # Light (N1 + N2)
            if label == 3: return 3  # Deep (N3)
            return label
        
        results_4cls = [map_5to4(r) for r in results]
        
        print(f"[推理] 映射后分布（4分类）:")
        print(f"  {Counter(results_4cls)}")
        
        # 计算统计指标
        total = len(results_4cls)
        stats = {
            "W_ratio": results_4cls.count(0) / total,
            "REM_ratio": results_4cls.count(1) / total,
            "Light_ratio": results_4cls.count(2) / total,
            "Deep_ratio": results_4cls.count(3) / total
        }
        
        # 简易质量分数
        # 公式：深睡 30% + REM 30% + 非清醒 40%
        quality_score = int((
            stats["Deep_ratio"] * 0.3 + 
            stats["REM_ratio"] * 0.3 + 
            (1 - stats["W_ratio"]) * 0.4
        ) * 100)
        
        return {
            "hypnogram": results_4cls,
            "hypnogram_raw": results,  # 保留原始 5 分类结果
            "stats": stats,
            "quality_score": quality_score
        }
    


def visualize_hypnogram(hypnogram, save_path=None):
    """
    简单的文本可视化
    """
    stage_names = ['W', 'REM', 'Light', 'Deep']
    stage_symbols = ['▁', '▃', '▅', '█']
    
    print("\n" + "="*80)
    print("睡眠阶梯图（每个字符 = 30秒）")
    print("="*80)
    
    # 每行显示 60 个 epoch（30 分钟）
    for i in range(0, len(hypnogram), 60):
        chunk = hypnogram[i:i+60]
        time_label = f"{i*0.5/60:.1f}h"
        symbols = ''.join([stage_symbols[s] for s in chunk])
        print(f"{time_label:>6s} | {symbols}")
    
    print("="*80)
    print(f"图例: {' | '.join([f'{stage_symbols[i]}={stage_names[i]}' for i in range(4)])}")
    print("="*80 + "\n")

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import numpy as np
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import os
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns


# --- 环境设置 ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
data_dir = os.path.join(os.getcwd(), 'sleep-edf', 'data')
model_save_path = os.path.join(data_dir, 'model', 'model021101.pth')
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)

# --- 超参数 ---
batch_size = 32
window_size = 15    # n
lr = 5e-4
num_epochs = 30
num_classes = 5

# --- 1. 载入原始的病人数据到内存 ---
def load_patient_data_to_dict(root_path):
    """
    只把每个病人的原始数据读入字典，不构建滑动窗口。
    内存占用将从 40GB 降到 约 1.5GB。
    """
    patient_dir = os.path.join(root_path, 'patient_data')
    patients = {}
    x_files = sorted([f for f in os.listdir(patient_dir) if f.endswith('_X.npy')])
    
    for f in tqdm(x_files, desc="Loading Patient Raw Data"):
        pid = f.replace('_X.npy', '')
        x = np.load(os.path.join(patient_dir, f)).astype(np.float32) # (N, 1, 3000)
        y = np.load(os.path.join(patient_dir, f"{pid}_y.npy")).astype(np.int64)
        
        # 预先清理无效标签
        valid = (y != -1)
        patients[pid] = {'x': x[valid], 'y': y[valid]}
        
    return patients


# --- 2. 数据集类（增强增广：噪声 + 随机遮罩，移除时间拉伸）---
class SleepContextDataset(Dataset):
    def __init__(self, patient_dict, window_size, augment=False):
        self.window_size = window_size
        self.half_window = window_size // 2
        self.augment = augment
        self.samples = []
        
        # 构建索引表：存储 (病人ID, 该中心Epoch的索引)
        for pid, data in patient_dict.items():
            n_epochs = len(data['y'])
            for i in range(self.half_window, n_epochs - self.half_window):
                self.samples.append((pid, i))
        
        self.patient_dict = patient_dict

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pid, center_idx = self.samples[idx]
        
        # 实时切片：获取窗口数据 (window_size, 1, 3000)
        x = self.patient_dict[pid]['x'][center_idx - self.half_window : center_idx + self.half_window + 1]
        y = self.patient_dict[pid]['y'][center_idx]
        
        x = torch.from_numpy(x).float()
        y = torch.tensor(y, dtype=torch.long)
        
        # 数据增强...
        if self.augment:
            if torch.rand(1) < 0.2:
                x = x + torch.randn_like(x) * 0.003
        
        return x, y


# --- 3. 模型（n通道输入）---
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ELU(),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch)
        )
        self.se = SEBlock(out_ch) # 新增注意力
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        self.elu = nn.ELU()

    def forward(self, x):
        out = self.conv(x)
        out = self.se(out) # 赋予特征通道权重
        return self.elu(out + self.shortcut(x))

# --- 修改后的主网络 ---
class SleepStageNetV8(nn.Module):
    def __init__(self, num_classes, window_size=5):
        super().__init__()
        self.window_size = window_size
        self.n_fft = 512
        self.hop_length = 64
        
        # --- 分支 A: 频谱特征提取 (2D CNN) ---
        self.spec_stem = nn.Sequential(
            nn.Conv2d(1, 32, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(32),
            nn.ELU()
        )
        self.spec_stage = nn.Sequential(
            ResBlock(32, 64, stride=2),
            ResBlock(64, 128, stride=2),
            ResBlock(128, 128, stride=2), # 缩减输出通道为128
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        # --- 分支 B: 原始波形特征提取 (1D CNN) ---
        # 模仿 TinySleepNet/DeepSleepNet 的小核+大核思想简化版
        self.raw_branch = nn.Sequential(
            # 第一层使用较大的卷积核捕捉低频节律 (如 Delta 波)
            nn.Conv1d(1, 64, kernel_size=64, stride=8, padding=32, bias=False),
            nn.BatchNorm1d(64),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=8, stride=8),
            # 第二层捕捉局部细节
            nn.Conv1d(64, 128, kernel_size=16, stride=1, padding=8, bias=False),
            nn.BatchNorm1d(128),
            nn.ELU(),
            nn.AdaptiveAvgPool1d(1) # 输出维度 (B*T, 128, 1)
        )
        
        # --- 融合与时序层 ---
        # 两个分支各出 128 维，合起来是 256
        self.rnn = nn.GRU(128 + 128, 128, num_layers=2, bidirectional=True, batch_first=True, dropout=0.3)
        
        # 分类器
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x: [Batch, n, 1, 3001]
        b, t, c, s = x.shape
        x_flat = x.view(b * t, c, s) # [B*T, 1, 3001]
        
        # --- 1. 频谱分支 (2D) ---
        # 在 forward 内部计算频谱
        win = torch.hann_window(self.n_fft).to(x.device)
        stft = torch.stft(x_flat.squeeze(1), n_fft=self.n_fft, hop_length=self.hop_length, 
                          window=win, return_complex=True, center=True)
        mag = (torch.abs(stft) + 1e-8).log().unsqueeze(1) # [B*T, 1, F, T]
        mag = mag[:, :, :128, :] # 取低频部分
        
        spec_fea = self.spec_stem(mag)
        spec_fea = self.spec_stage(spec_fea).flatten(1) # [B*T, 128]
        
        # --- 2. 原始波形分支 (1D) ---
        raw_fea = self.raw_branch(x_flat).flatten(1) # [B*T, 128]
        
        # --- 3. 特征融合 ---
        combined_fea = torch.cat([spec_fea, raw_fea], dim=1) # [B*T, 256]
        
        # --- 4. 还原序列维度过 RNN ---
        combined_fea = combined_fea.view(b, t, -1) # [B, n, 256]
        rnn_out, _ = self.rnn(combined_fea)  # [B, n, 256]
        
        # --- 5. 取中间 Epoch 输出 ---
        mid_idx = t // 2
        return self.classifier(rnn_out[:, mid_idx, :])



# --- 4. 评估 ---
def evaluate(model, loader):
    model.eval()
    all_p_4, all_l_4 = [], [] # 存放合并后的 4 分类结果
    
    # 定义 5类 到 4类 的映射规则 (根据你的实际标签定义)
    # 原始(5类): 0:W, 1:N1, 2:N2, 3:N3, 4:REM
    # 目标(4类): 0:W, 1:REM, 2:Light(N1+N2), 3:Deep(N3)
    def map_5to4(label):
        if label == 0: return 0 # W -> W
        if label == 4: return 1 # REM -> REM
        if label == 1 or label == 2: return 2 # N1, N2 -> Light
        if label == 3: return 3 # N3 -> Deep
        return label
    with torch.no_grad():
        for x, y in loader:
            outputs = model(x.to(device)) # 输出是 5 维
            preds_5 = outputs.argmax(1).cpu().numpy()
            labels_5 = y.numpy()
            
            # 执行合并逻辑
            preds_4 = [map_5to4(p) for p in preds_5]
            labels_4 = [map_5to4(l) for l in labels_5]
            
            all_p_4.extend(preds_4)
            all_l_4.extend(labels_4)
            
    return accuracy_score(all_l_4, all_p_4), f1_score(all_l_4, all_p_4, average='macro'), \
           confusion_matrix(all_l_4, all_p_4), Counter(all_p_4)

def print_confusion_matrix(cm, acc, f1, epoch):
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Epoch {epoch} | Acc: {acc:.4f} | F1: {f1:.4f}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()
    # 计算每个分类的准确率
    class_accuracies = cm.diagonal() / cm.sum(axis=1)
    print(f"Epoch {epoch} | Overall Acc: {acc:.4f} | F1: {f1:.4f}")
    print("Class-wise Accuracies:")
    for i, class_acc in enumerate(class_accuracies):
        print(f"  Class {i}: {class_acc:.4f}")

In [9]:
# from predictor_v1 import SleepPredictorV1
# from train1 import SleepStageNetV8

# 初始化
predictor = SleepPredictorV1(
    model_path='./sleep-edf/data/model/model021101.pth',
    device='cuda',
    window_size=15
)

# 加载模型
predictor.load_model(SleepStageNetV8)

# 预测
epochs = predictor.preprocess('./sleep-edf/data/test/SC4031E0-PSG.edf')
result = predictor.predict(epochs)

# 可视化
visualize_hypnogram(result['hypnogram'])


[初始化] 使用设备: cuda
[初始化] 窗口大小: 15
[加载模型] 路径: ./sleep-edf/data/model/model021101.pth
[加载模型] 成功加载，参数量: 1,368,421

[预处理] 开始处理: ./sleep-edf/data/test/SC4031E0-PSG.edf


C:\Users\Eason\AppData\Local\Temp\ipykernel_1872\4178500581.py:50: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
C:\Users\Eason\AppData\Local\Temp\ipykernel_1872\4178500581.py:50: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
C:\Users\Eason\AppData\Local\Temp\ipykernel_1872\4178500581.py:50: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


[预处理] 原始采样率: 100.0 Hz
[预处理] 原始通道: ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'Resp oro-nasal', 'EMG submental', 'Temp rectal', 'Event marker']
[预处理] 原始时长: 23.50 小时
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
[预处理] 选择通道: EEG Fpz-Cz
[预处理] 滤波: 0.3-35 Hz
[预处理] 总采样点: 8460000
[预处理] Epoch 数量: 2820
[预处理] 输出形状: (2820, 1, 3000)

[推理] 开始预测，Epoch 数量: 2820
[推理] 进度: 100/2820
[推理] 进度: 200/2820
[推理] 进度: 300/2820
[推理] 进度: 400/2820
[推理] 进度: 500/2820
[推理] 进度: 600/2820
[推理] 进度: 700/2820
[推理] 进度: 800/2820
[推理] 进度: 900/2820
[推理] 进度: 1000/2820
[推理] 进度: 1100/2820
[推理] 进度: 1200/2820
[推理] 进度: 1300/2820
[推理] 进度: 1400/2820
[推理] 进度: 1500/2820
[推理] 进度: 1600/2820
[推理] 进度: 1700/2820
[推理] 进度: 1800/2820
[推理] 进度: 1900/2820
[推理] 进度: 2000/2820
[推理] 进度: 2100/2820
[推理] 进度: 2200/2820
[推理] 进度: 2300/2820
[推理] 进度: 2400/2820
[推理] 进度: 2500/2820
[推理] 进度: 2600/2820
[推理] 进度: 2700/2820
[推理] 进度: 2800/2820
[推理] 完成，原始预测分布（5分类）:
  Counter({0: 1964, 2: 452, 4: 182, 3: 138, 1: 84})
[推理] 映射后分布（4分类）:
  

1. 测试比较成功，可开始前后端标准化
2. 可试用Vue-假数据样板设计先？
3. 关于可以画两个图 其一：20小时的完整预测 其二：预测完成后截取其中的睡眠阶段进行展示
4. 前端应用页面企业化
